# Ignite-3B Session S01 - Baseline Cond A + weco Cond B kick

**Goal**: eval v_0 (Qwen2.5-3B base) on Ignite benches + start weco Cond B run.

Setup:
1. Accelerator = GPU T4 x2
2. Internet ON
3. Persistence = Variables and Files
4. Kaggle Secret HF_TOKEN (write) + ANTHROPIC_API_KEY (Cond B)

Runs Cond A eval, uploads results dataset, kicks weco run in bg.

In [ ]:
BASE_MODEL = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
OUTPUT_DATASET = 'vitorscrt/ignite-3b-baseline-s01'
BENCHES_A = ['omni_math', 'livecodebench', 'matharena', 'aime']
print(f'base={BASE_MODEL} -> {OUTPUT_DATASET}')

In [ ]:
!pip install -q -U 'transformers>=4.46.0' 'peft>=0.13.0' 'datasets>=3.0.0' 'accelerate>=1.0.0' 'unsloth>=2025.1.0' 'trl>=0.12.0' 'vllm>=0.6.0' 'math-verify>=0.5.2' 'latex2sympy2' 'sympy' 'weco>=0.3.40' 'scipy' kaggle huggingface_hub

In [ ]:
import os, subprocess
if not os.path.exists('/kaggle/working/caracal-1'):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', 's07-hybrid-agentic',
                    'https://github.com/iterate-labs-ai/caracal-1.git',
                    '/kaggle/working/caracal-1'], check=True)
os.chdir('/kaggle/working/caracal-1')
rev = subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip()
print(f'HEAD={rev}')

In [ ]:
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
hf_token = sec.get_secret('HF_TOKEN')
from huggingface_hub import login
login(token=hf_token)

In [ ]:
import subprocess
for script in ['build_omni_math', 'build_livecodebench', 'build_matharena', 'build_aime']:
    subprocess.run(['python', '-m', f'data.ignite.{script}'], check=False)
subprocess.run(['ls', '-la', 'data/ignite/'], check=False)

In [ ]:
import subprocess, sys
os.makedirs('results', exist_ok=True)
subprocess.run([sys.executable, '-m', 'eval.ignite.run_all_ignite',
                '--base', BASE_MODEL,
                '--benches', *BENCHES_A,
                '--out', 'results/cond_A.json'], check=True)

In [ ]:
import json
res = json.load(open('results/cond_A.json'))
for k, v in res.items():
    if isinstance(v, dict) and 'accuracy' in v:
        print(f"{k:20s}: {v['accuracy']*100:5.1f}%  n={v.get('n', 0)}")

In [ ]:
import json, subprocess
from pathlib import Path
pub_dir = Path('results')
(pub_dir / 'dataset-metadata.json').write_text(json.dumps({
    'title': 'Ignite-3B Baseline S01',
    'id': OUTPUT_DATASET,
    'licenses': [{'name': 'Apache-2.0'}],
}, indent=2))
r = subprocess.run(['kaggle', 'datasets', 'create', '-p', str(pub_dir), '--public'],
                   capture_output=True, text=True, check=False)
print(r.stdout, r.stderr)
if r.returncode != 0:
    subprocess.run(['kaggle', 'datasets', 'version', '-p', str(pub_dir),
                    '-m', 'ignite baseline s01'], check=True)
print(f'-> {OUTPUT_DATASET}')